In [2]:
pip install llama-cpp-python

Defaulting to user installation because normal site-packages is not writeable
  Using cached llama_cpp_python-0.3.16.tar.gz (50.7 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
Using cached diskcache-5.6.3-py3-none-any.whl (45 kB)
Failed to build llama-cpp-python
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Building wheel for llama-cpp-python (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [19 lines of output]
      *** scikit-build-core 0.11.6 using CMake 4.2.1 (wheel)
      *** Configuring CMake...
      loading initial cache file C:\Users\USER\AppData\Local\Temp\tmp2dem_5ny\build\CMakeInit.txt
      -- Building for: NMake Makefiles
      CMake Error at CMakeLists.txt:3 (project):
        Running
      
         'nmake' '-?'
      
        failed with:
      
         no such file or directory
      
      
      CMake Error: CMAKE_C_COMPILER not set, after EnableLanguage
      CMake Error: CMAKE_CXX_COMPILER not set, after EnableLanguage
      -- Configuring incomplete, errors occurred!
      
      *** CMake configuration failed
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for llama-cpp-python
ERROR: Failed to build installabl

In [3]:
pip install llama-index

Defaulting to user installation because normal site-packages is not writeable
  Using cached setuptools-80.10.1-py3-none-any.whl.metadata (6.7 kB)
  Using cached openai-2.15.0-py3-none-any.whl.metadata (29 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached jiter-0.12.0-cp313-cp313-win_amd64.whl.metadata (5.3 kB)
  Using cached pydantic_core-2.41.5-cp313-cp313-win_amd64.whl.metadata (7.4 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
INFO: pip is looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of llama-cloud-services to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter 

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account al

In [4]:
pip install transformers accelerate

Defaulting to user installation because normal site-packages is not writeable
  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.12.0-py3-none-any.whl (380 kB)
Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
from llama_cpp import Llama

llm = Llama(
    model_path = "./models/mistral-7b-instruct-v0.1.Q4_K_M.gguf",
    n_ctx=2048,
    n_threads=4,
    n_gpu_layers=0
)

def local_llm(prompt:  str) -> str:
    output = llm(prompt, max_tokens=500, stop=["</s>"])
    return output['choices'][0]['text'].strip()



ModuleNotFoundError: No module named 'llama_cpp'


 This ense the Ai angents operate entirely offline, cost-free
 Fully under our control

In [ ]:
def local_llm(prompt:  str) -> str:
    output = llm(prompt, max_tokens=500, stop=["</s>"])
    return output['choices'][0]['text'].strip()




### Step 2: Define Custom Wrapper for CrewAI

 Wrapping llm in class to compactible to CrewAi

In [ ]:
class LocalLLMWrapper:
    def __init__(self, engine):
        self.engine = engine

    def complete(self, prompt: str) -> str:
        return self.engine(prompt)

llm_wrapper = LocalLLMWrapper(local_llm)


# Step 3: Define Agents wih Roles, Goals, and Backstories

In [ ]:


from crewai import Agent
researcher = Agent(
    role='Market Research Analyst',
    goal='Analyze competitors and summarize their marketing strategies',
    backstory='An expert in market intelligence and competitive analysis.',
    llm=llm_wrapper,
    allow_delegation=False
)

In [ ]:
writer = Agent(
    role='Content Strategist',
    goal='Use research to create a compelling marketing strategy document',
    backstory='A seasoned content strategist with a flair for storytelling.',
    llm=llm_wrapper
)


### These agents will collaborate later n a shared task, and defining them properly
now lays the foundation for effective multi-agent coordination
 


### Step 4: Define Tasks for Each Agent

- In CrewAI, tasks are actionable units work thst agents are responsible for. Each includes a description of
- What needs to be done, the agent responsible, and the expected output.
- We can also define dependencies between tasks.
- Allowing one agent to build upon the output of another.
- This especially useful for multi-step workflows.
 

In [ ]:
from crewai import Task

task1 = Task(
    description = ""List top 3 competitors and their marketing strtegies based on current trends.",
    agent=researcher,
    expected_output="A summary of 3 competitors with key market strategies."
)

task2 = Task(
    description="Create a content marketing strategy based on the competitor summary.",
    agent=writer,
    expected_output="A structured document with our content strategy inspired by competitors.",
    depends_on=[task1]
)

This structure enbetween sures a clear flow of information and coordination agents.

### Step 5: Create and Run a Crew

A Crew in CrewAi is essentially the manager who coordinates how agents interact and execute their assigned tasks.

By passing in our list of agents and tasks, we will define a collaborative environment where each agent knws what to do and when:

In [ ]:
from crewai import Crew

crew = Crew(
    agents=[researcher, writer],
    tasks=[task1, task2],
    verbose=True # see what each agent does
)

result = crew.kickoff()
print(result)

Setting `verbose=True` allows us to observe how each agent reasons through
its task in real-time, making debugging and optimization easier.
The `kickoff()` method then launches the multi-agent workflow and returns
the final output after all tasks have beeen completed